# K-Means Clustering

This model clusters the *regions* into k-groups to the nearest randomly generated centroid using Euclidean distance.

In [2]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
from methods.read_data import createMatrix, printMatrix
from methods.normalize_data import normalizeData, standardizeData, coalesceData
from models.clustering import k_means_clustering
from evaluation.conf_matrix import evaluate
from pathlib import Path
from os import path
from copy import deepcopy
import csv

17 regions x 19 fields
showing fields 0 to 18 

  [0] Table_2 | Poverty Incidence among Population (%) 2023
  [1] Table_2 | Poverty Incidence among Population (%) 2021
  [2] Table_2 | Poverty Incidence among Population (%) 2018
  [3] Table_12 | Estimate (%) 2023
  [4] Table_6a | Estimate ('000) 2023
  [5] Table_2 | Annual Per Capita Poverty Threshold (in PhP) 2023
  [6] Total Family Size | Average Income
  [7] Total Family Size | Average Expenditure
  [8] Getting information about goods or services Percentage
  [9] Getting information from general government organizations Percentage
  [10] Interacting with government organizations Percentage
  [11] Seeking health-related information Percentage
  [12] Making an appointment with a health practitioner via the internet Percentage
  [13] Attending an online class/online course (on any subject) Percentage
  [14] online encyclopedias Percentage
  [15] websites for a job Percentage
  [16] DERIVED | Change in poverty incidence 2021 to 2023 (per

### Step 1: Create the Dataset from the `PROJECT_DATASET` folder
- This will create a matrix with 19 fields/attributes and 17 regions

In [4]:
data_set_dir = path.join(path.abspath("."), "PROJECT_DATASET")          # Get the path to the dataset folder
regions, fields, matrix = createMatrix(Path(data_set_dir))              # Create the matrix from the dataset
printMatrix(regions, fields, matrix, max_regions=18, max_fields=20)     # Print the matrix with a maximum of 18 regions and 20 fields

17 regions x 19 fields
showing fields 0 to 18 

  [0] Table_2 | Poverty Incidence among Population (%) 2023
  [1] Table_2 | Poverty Incidence among Population (%) 2021
  [2] Table_2 | Poverty Incidence among Population (%) 2018
  [3] Table_12 | Estimate (%) 2023
  [4] Table_6a | Estimate ('000) 2023
  [5] Table_2 | Annual Per Capita Poverty Threshold (in PhP) 2023
  [6] Total Family Size | Average Income
  [7] Total Family Size | Average Expenditure
  [8] Getting information about goods or services Percentage
  [9] Getting information from general government organizations Percentage
  [10] Interacting with government organizations Percentage
  [11] Seeking health-related information Percentage
  [12] Making an appointment with a health practitioner via the internet Percentage
  [13] Attending an online class/online course (on any subject) Percentage
  [14] online encyclopedias Percentage
  [15] websites for a job Percentage
  [16] DERIVED | Change in poverty incidence 2021 to 2023 (per

### Step 2: Cleanup the data
- Convert the matrix into a dictionary of region statistics with region id as key and list of attributes as values
- Coalesce missing values to 0 for clustering, then standardize it using Z-score Scaling

In [5]:

# Coalesce NA values to 0 for clustering, then standardize it.
cleaned_data = standardizeData(coalesceData(matrix))

# Create dictionary of region statistics with region id as key and list of features as value
region_stats = {regions[i]: cleaned_data[i] for i in range(len(regions))}


### Step 3: Run the K-means clustering with k=5
- Run the k-means clustering with k set to 5
- Then display the categories for each region in the Dataset

In [6]:
copied_region_stats = deepcopy(region_stats)
categories = k_means_clustering(copied_region_stats, 5)
# For each labeled dataset, it includes the (1) label/cluster, (2) player id, (3) name of the player, and (4) data points associated with the player id
print("\nLabeled Dataset:")
for id, points in region_stats.items():
  region_name = regions[regions.index(id)]
  print(f"{categories[id]:<1} : {id:<12} - {region_name:<25} {points}")

Initial Centroids
(-0.0730346428460758, 0.7177006379019215, -0.33932698551546053, -0.09588585722798244, 0.4697466582960525, 0.5291654118596593, 0.1067893733550982, -0.2415249964955562, 0.04018653165947404, 0.21397618485588887, 0.6072924765978324, -0.1780759655566264, 0.6959405189820638, -0.2816933423473088, 1.0103492451909994, 1.0244992576712384, -2.051967393621597, 0.5227786869406263, -0.4182139188531235)
(-0.6716578761737317, -0.724125889807937, -0.8038083397145457, -0.6392390481865505, -0.6872820456258393, 0.49100836866515174, 0.2913714839987801, 0.03477265173231591, 0.025650977654983594, -0.8155318743564045, -0.11403308512535665, -0.9777378486222298, 0.9992992067434762, 0.9155033626287524, 0.345474258033051, -0.9703407066049159, 0.1619534883493043, 0.6308302088614852, -0.2566312683871441)
(-1.791305034805088, -1.914725067992593, -1.3806642150908288, -1.544827699784164, -1.5282247748973126, 1.7463365472965782, 2.807913310915739, 2.9609200560387405, 2.1042352002971616, 1.906944993338

### Step 4: Evaluate the clusters using a Confusion Matrix
- Read the actual class of each region from the tagged dataset `region_policy_results.csv`
- K-means only gives a cluster number, so each cluster is labeled with the most common actual class among its regions (majority vote)
- Compare the predicted class with the actual class and compute the Recall, Precision, F1 Score and Accuracy for each class (one vs. rest)

In [7]:
# Read the actual class of each region from the tagged dataset
tagged_dir = path.join(data_set_dir, "region_policy_results.csv")      # Get the path to the tagged dataset
with open(tagged_dir, newline="", encoding="utf-8-sig") as file:
  actual_classes = {row["Region"]: row["Class"] for row in csv.DictReader(file)}

# Label each cluster with the most common actual class among its regions (majority vote)
cluster_labels = {}
for cluster in set(categories.values()):
  cluster_classes = [actual_classes[id] for id in categories if categories[id] == cluster]
  cluster_labels[cluster] = max(cluster_classes, key=cluster_classes.count)

# Display the class assigned to each cluster
print("Cluster Labels:")
for cluster, label in sorted(cluster_labels.items()):
  print(f"{cluster:<1} : {label}")

# Create the list of actual and predicted classes for each region
actual = [actual_classes[id] for id in regions]                       # Actual class from the tagged dataset
predicted = [cluster_labels[categories[id]] for id in regions]        # Predicted class from the cluster label

print("\nActual vs. Predicted:")
for i in range(len(regions)):
  print(f"{regions[i]:<12} - {actual[i]:<9} {predicted[i]}")
print()

# Calculate and display the confusion matrix and the performance metrics for each class
results = evaluate(actual, predicted)

Cluster Labels:
0 : RETAIN
1 : RETAIN
2 : DECREASE
3 : INCREASE
4 : RETAIN

Actual vs. Predicted:
NCR          - DECREASE  DECREASE
CAR          - RETAIN    RETAIN
Region I     - RETAIN    RETAIN
Region II    - RETAIN    RETAIN
Region III   - RETAIN    RETAIN
Region IV-A  - DECREASE  RETAIN
MIMAROPA     - INCREASE  INCREASE
Region V     - INCREASE  INCREASE
Region VI    - RETAIN    RETAIN
Region VII   - RETAIN    RETAIN
Region VIII  - INCREASE  INCREASE
Region IX    - INCREASE  INCREASE
Region X     - INCREASE  INCREASE
Region XI    - INCREASE  INCREASE
Region XII   - INCREASE  INCREASE
Region XIII  - INCREASE  RETAIN
BARMM        - INCREASE  INCREASE

Confusion Matrix (Positive class: DECREASE)
                      Predicted    
                  Positive Negative
Actual    Positive       1        1
          Negative       0       15
Recall:    0.5000
Precision: 1.0000
F1 Score:  0.6667
Accuracy:  0.9412

Confusion Matrix (Positive class: INCREASE)
                      Predicted   